# Neural VAD — training notebook

The fourth detector in [vad-from-scratch](https://github.com/), and the first one
nobody tuned by hand. Energy, zero-crossing and spectral all reduce a frame to a
single number and threshold it; this one learns what to reduce it to.

The output is an ONNX graph plus a small JSON of front-end parameters, which
`backend/app/vad/neural.py` loads. Everything in the export is pinned to the
frame grid the rest of the project already uses — **30 ms window, 10 ms hop, 16
kHz** — so the probability curve lands on the same time axis as the other three
detectors and the explorer can plot them against each other.

### Before you run

1. **Settings → Accelerator → GPU P100** (or T4 x2 — only one is used).
2. **Settings → Internet → On**, if you want the Silero label cross-check. Skip
   it and the notebook falls back to energy labels alone.
3. **Add Input → Datasets**, at least one from each of the first two rows:

| Role   | Slug                                                     | Size   | Needed                    |
| ------ | -------------------------------------------------------- | ------ | ------------------------- |
| speech | `bacnguyenne/librispeech-train-clean-100`                | ~6 GB  | yes                       |
| noise  | `dogrose/musan-dataset`                                  | ~11 GB | strongly recommended      |
| noise  | `chrisfilo/urbansound8k` or any ESC-50 / AudioSet mirror | ~6 GB  | optional                  |
| rir    | `tunguz/bird-big-impulse-response-dataset`               | ~1 GB  | optional, helps far-field |

The loader reads whatever is mounted and adapts. Nothing is hardcoded to a
particular slug — datasets are assigned a role from their name, and anything
unrecognised is printed and skipped rather than guessed at, because a speech
corpus misfiled as noise would silently poison the labels.

### What it costs

Roughly 25 min packing, then 3–5 h training, well inside a single session.
`cfg.time_budget_h` stops training and runs the export regardless, so a session
that turns out slower than expected still produces a usable model. There is also
a `SMOKE` switch in the config cell that runs the whole thing end to end in about
15 minutes, which is worth doing once before committing a real run.


In [ ]:
import os
import gc
import json
import math
import time
import pickle
import random
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from scipy.signal import resample_poly, fftconvolve

warnings.filterwarnings("ignore")
torch.backends.cudnn.benchmark = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"torch {torch.__version__} | torchaudio {torchaudio.__version__} | {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
print(f"{os.cpu_count()} cpus")

## Configuration

Two numbers here are not free parameters. `win = 480` and `hop = 160` are 30 ms
and 10 ms at 16 kHz, which is exactly what `DecisionSettings` defaults to in
`backend/app/vad/pipeline.py`. Matching them means the backend can hand this
model the frame matrix it already builds for every other detector, instead of
re-buffering the audio on a second grid — and it means frame _t_ here and frame
_t_ in the energy detector cover the same samples, so the explorer's plots line
up.

`n_fft` is set to `win` rather than rounded up to 512. A 512-point transform
would make `torch.stft` zero-pad the 480-sample window symmetrically out to 512
and analyse a 512-sample frame, which is a different frame from the one
`frame_signal` produces. 480 factors as 2⁵·3·5, so the transform is still fast.


In [ ]:

SMOKE = False


@dataclass
class Cfg:
    sr: int = 16000
    win: int = 480          # 30 ms  win = windows
    hop: int = 160          # 10 ms  hop = hop size
    n_fft: int = 480        # 30 ms
    n_mels: int = 64
    fmin: int = 20
    fmax: int = 7600

    chunk_frames: int = 400          # 4.0 s scenes
    batch: int = 128
    epochs: int = 20
    steps_per_epoch: int = 1500
    val_chunks: int = 6144
    lr: float = 3e-3

    weight_decay: float = 1e-2
    pct_warmup: float = 0.08
    ema_decay: float = 0.999
    label_smooth: float = 0.02
    boundary_weight: float = 0.3
    boundary_frames: int = 2
    grad_clip: float = 5.0
    time_budget_h: float = 7.5       # stop training when time budget is reached (hours)

    # Modal
    channels: int = 192
    kernels: tuple = (11, 13, 17, 21, 25)
    sub_blocks: int = 2
    dropout: float = 0.1
    gru_hidden: int = 192

    # Corpus of speech/ noise
    speech_hours: float = 35.0
    noise_hours: float = 20.0
    max_rirs: int = 3000
    min_speech_sec: float = 1.0

    # Scene synthesis
    snr_range: tuple = (-10.0, 25.0)
    level_range: tuple = (-38.0, -12.0)
    gap_range: tuple = (0.08, 1.6)
    span_range: tuple = (0.3, 2.5)
    p_reverb: float = 0.30
    p_noise_only: float = 0.12
    p_dense: float = 0.12
    p_digital_silence: float = 0.03
    p_synthetic_noise: float = 0.08
    p_lowpass: float = 0.12
    p_clip: float = 0.05

    # Label cross-check
    use_teacher: bool = True
    teacher_frac: float = 0.25
    min_label_iou: float = 0.55

    cache_dir: str = "/kaggle/temp/vadcache"
    out_dir: str = "/kaggle/working"
    input_dir: str = "/kaggle/input"
    workers: int = 4
    seed: int = 1234

    @property
    def chunk_samples(self):
        return (self.chunk_frames - 1) * self.hop + self.win

    @property
    def n_freqs(self):
        return self.n_fft // 2 + 1


cfg = Cfg()
cfg.workers = max(1, min(4, os.cpu_count() or 4))

if SMOKE:
    cfg.epochs, cfg.steps_per_epoch, cfg.val_chunks = 2, 120, 1024
    cfg.speech_hours, cfg.noise_hours, cfg.max_rirs = 2.0, 1.5, 200
    cfg.teacher_frac, cfg.time_budget_h = 0.05, 0.5

if not Path(cfg.input_dir).exists():
    cfg.cache_dir, cfg.out_dir = "./vadcache", "./out"

Path(cfg.cache_dir).mkdir(parents=True, exist_ok=True)
Path(cfg.out_dir).mkdir(parents=True, exist_ok=True)

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)

VAL_SNRS = [-5.0, 0.0, 5.0, 10.0, 20.0, 99.0]
SNR_NAMES = ["-5 dB", "0 dB", "5 dB", "10 dB", "20 dB", "clean"]

print(f"chunk {cfg.chunk_samples} samples ({cfg.chunk_samples / cfg.sr:.2f} s) "
      f"-> {cfg.chunk_frames} frames")
print(f"planned {cfg.epochs * cfg.steps_per_epoch} steps, budget {cfg.time_budget_h} h"
      + ("   [SMOKE]" if SMOKE else ""))

## Finding the data

Every mounted dataset gets a role from its slug. Anything unrecognised is
printed and skipped rather than guessed at — put it in `MANUAL_ROLES` and rerun
if the guess was wrong.

MUSAN ships a `speech/` folder alongside its noise. Those files are excluded
here: babble treated as noise would contradict the labels, since the training
scenes assume the noise bed contains no speech when computing SNR over
speech-active frames.


In [ ]:
MANUAL_ROLES = {}          # {"dataset-folder-name":"speech"|"noise" |"rir"}

SPEECH_HINTS = ("librispeech", "libri", "common_voice", "commonvoice", "common-voice", 
                "cv-valid", "cv-corpus", "vctk", "libritts", "voxpopuli", "gigaspeech",
                "tedlium", "timit")
NOISE_HINTS = ("musan", "audioset", "dns", "esc50", "esc-50", "urbansound", "noise",
               "fsd50k", "fsd", "freesound", "demand", "wham", "chime", "environmental")
RIR_HINTS = ("rir", "impulse", "bird-big", "reverb", "aachen")
AUDIO_EXT = {".wav", ".flac", ".mp3", ".ogg", ".opus", ".m4a"}


def scan_audio(root, limit=400_000):
    found, stack = [], [str(root)]
    while stack and len(found) < limit:
        try:
            entries = list(os.scandir(stack.pop()))
        except OSError:
            continue
        for e in entries:
            if e.is_dir(follow_symlinks=False):
                stack.append(e.path)
            elif os.path.splitext(e.name)[1].lower() in AUDIO_EXT:
                found.append(e.path)
    return found


def role_of(name):
    if name in MANUAL_ROLES:
        return MANUAL_ROLES[name]
    n = name.lower()
    for hints, role in ((RIR_HINTS, "rir"), (NOISE_HINTS, "noise"), (SPEECH_HINTS, "speech")):
        if any(h in n for h in hints):
            return role
    return None


def dataset_roots(base, max_depth=4):
    level = [Path(base)]
    for _ in range(max_depth + 1):
        children = []
        for directory in level:
            try:
                children.extend(sorted(c for c in directory.iterdir() if c.is_dir()))
            except OSError:
                continue
        if not children:
            break
        matched = [c for c in children if role_of(c.name) is not None]
        if matched:
            return matched, [c for c in children if c not in matched]
        level = children
    return [], level[:20]


def discover(base):
    pools = {"speech": [], "noise": [], "rir": []}
    base = Path(base)
    if not base.exists():
        print(f"{base} not found - set cfg.input_dir or fill MANUAL_ROLES")
        return pools

    roots, unknown = dataset_roots(base)
    if not roots:
        print(f"nothing under {base} looked like a known corpus. Directories seen:")
        for directory in unknown:
            print(f"    {directory.relative_to(base)}")
        print("Add one of those names to MANUAL_ROLES and rerun.")
        return pools

    for ds in roots:
        role = role_of(ds.name)
        files = scan_audio(ds)
        if not files:
            print(f"  {ds.relative_to(base)}: no audio found")
            continue
        if role == "noise":
            files = [f for f in files if os.sep + "speech" + os.sep not in f.lower()]
        pools[role].extend(files)
        print(f"  {role:6s} {ds.relative_to(base)}  ->  {len(files)} files")

    for directory in unknown:
        print(f"  skipped {directory.relative_to(base)} - unknown role")
    return pools


print("scanning inputs")
pools = discover(cfg.input_dir)
for k, v in pools.items():
    random.Random(cfg.seed).shuffle(v)
    print(f"{k}: {len(v)} files")

if not pools["speech"]:
    raise RuntimeError("no speech corpus mounted - attach at least one speech dataset")
if not pools["noise"]:
    print("no noise corpus found - falling back to synthetic noise only, "
          "which will hurt the false alarm rate badly")

pools["rir"] = pools["rir"][:cfg.max_rirs]

## Labelling

There is no hand-labelled corpus here, and that is deliberate rather than a
compromise. Ground truth is taken from the _clean_ source before any noise is
mixed in, so the label stays exact no matter how bad the SNR gets afterwards.
Labelling the mixture instead would cap the model at the accuracy of whatever
labelled it.

The labeller itself is the energy detector from `backend/app/vad/energy.py`,
give or take: hysteresis around a threshold placed above the noise floor, then
gaps under 100 ms closed, blips under 60 ms dropped, boundaries dilated by 30 ms
to cover low-energy onsets.

The one addition is a zero-crossing rescue. Energy alone clips unvoiced
fricatives — the trailing /s/ that the rule-based pages keep having to buy back
with hangover — so quiet frames _next to_ detected speech that have a high
crossing rate get pulled in as well. Files whose energy spread is too small to
label confidently are dropped entirely rather than labelled badly.


In [ ]:
def frames_of(x, win, hop):
    n = 1 + (len(x) - win) // hop
    if n < 1:
        return np.zeros((0, win), np.float32)
    return np.lib.stride_tricks.as_strided(
        x, (n, win), (x.strides[0] * hop, x.strides[0]), writeable=False)


def hysteresis(v, hi, lo):
    out = np.zeros(len(v), bool)
    on = False
    for i in range(len(v)):
        on = v[i] > hi if not on else v[i] > lo
        out[i] = on
    return out


def runs(mask):
    d = np.diff(np.concatenate(([0], mask.astype(np.int8), [0])))
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)


def close_gaps(mask, k):
    s, e = runs(mask)
    for i in range(1, len(s)):
        if s[i] - e[i - 1] <= k:
            mask[e[i - 1]:s[i]] = True
    return mask


def drop_short(mask, k):
    s, e = runs(mask)
    for a, b in zip(s, e):
        if b - a < k:
            mask[a:b] = False
    return mask


def dilate(mask, k):
    if k <= 0:
        return mask
    out = mask.copy()
    s, e = runs(mask)
    for a, b in zip(s, e):
        out[max(0, a - k):b + k] = True
    return out


def energy_labels(x, cfg):
    fr = frames_of(x, cfg.win, cfg.hop)
    if len(fr) < 20:
        return None
    e = 10.0 * np.log10((fr ** 2).mean(1) + 1e-10)
    zcr = (np.diff(np.sign(fr), axis=1) != 0).mean(1)

    floor, peak = np.percentile(e, 10), np.percentile(e, 97)
    if peak - floor < 12.0:
        return None

    thr = max(floor + 7.0, peak - 32.0)
    lab = hysteresis(e, thr + 2.0, thr - 3.0)
    if not lab.any():
        return None

    near = dilate(lab, 6)
    lab |= near & (e > floor + 4.0) & (zcr > 0.22)
    lab = close_gaps(lab, int(0.10 * cfg.sr / cfg.hop))
    lab = drop_short(lab, int(0.06 * cfg.sr / cfg.hop))
    lab = dilate(lab, int(0.03 * cfg.sr / cfg.hop))
    return lab

## Packing

Decoding FLAC on four vCPUs cannot keep a GPU fed while also synthesising
scenes, so everything is decoded once into flat int16 shards and read back
through a memmap during training. Labels are computed in the same pass, packed
to bits, and stored with the segment list.

The shards go in `/kaggle/temp`, not `/kaggle/working`. A fresh session repacks
them, but the 20 GB working quota stays free for checkpoints and the export.


In [ ]:
def load_audio(path, sr):
    try:
        x, fs = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        try:
            w, fs = torchaudio.load(path)
            x = w.mean(0).numpy()
        except Exception:
            return None
    if x is None or len(x) == 0:
        return None
    if x.ndim > 1:
        x = x.mean(1)
    if fs != sr:
        g = math.gcd(int(fs), sr)
        x = resample_poly(x, sr // g, int(fs) // g).astype(np.float32)
    return np.ascontiguousarray(x, dtype=np.float32)


def pack_shard(job):
    wid, files, kind, budget, cfg = job
    path = os.path.join(cfg.cache_dir, f"{kind}_{wid}.raw")
    meta, off = [], 0
    min_len = int(cfg.min_speech_sec * cfg.sr)

    with open(path, "wb") as fh:
        for p in files:
            if off >= budget:
                break
            x = load_audio(p, cfg.sr)
            if x is None or len(x) < min_len:
                continue
            peak = np.abs(x).max()
            if peak < 1e-4:
                continue
            x = x * (0.95 / peak)

            lab = None
            if kind == "speech":
                lab = energy_labels(x, cfg)
                if lab is None or lab.mean() < 0.05:
                    continue
                nf = len(lab)
                x = x[:(nf - 1) * cfg.hop + cfg.win]

            fh.write(np.clip(x * 32767.0, -32768, 32767).astype(np.int16).tobytes())
            rec = {"w": wid, "o": off, "n": len(x)}
            if lab is not None:
                s, e = runs(lab)
                rec["seg"] = list(zip(s.tolist(), e.tolist()))
                rec["nf"] = len(lab)
                rec["lab"] = np.packbits(lab).tobytes()
            meta.append(rec)
            off += len(x)
    return meta


def build_pack(kind, files, hours, cfg):
    stamp = f"{kind}_{int(hours * 10)}_{len(files)}_{cfg.win}_{cfg.hop}"
    idx_path = os.path.join(cfg.cache_dir, f"{kind}_index.pkl")
    if os.path.exists(idx_path):
        with open(idx_path, "rb") as fh:
            saved = pickle.load(fh)
        if saved["stamp"] == stamp:
            print(f"{kind}: reusing cache ({len(saved['recs'])} files)")
            return saved["recs"]

    budget = int(hours * 3600 * cfg.sr / cfg.workers)
    jobs = [(w, files[w::cfg.workers], kind, budget, cfg) for w in range(cfg.workers)]
    t0 = time.time()
    with ProcessPoolExecutor(cfg.workers) as ex:
        recs = [r for part in ex.map(pack_shard, jobs) for r in part]

    total = sum(r["n"] for r in recs) / cfg.sr / 3600
    print(f"{kind}: {len(recs)} files, {total:.1f} h, {time.time() - t0:.0f}s")
    with open(idx_path, "wb") as fh:
        pickle.dump({"stamp": stamp, "recs": recs}, fh)
    return recs


speech_recs = build_pack("speech", pools["speech"], cfg.speech_hours, cfg)
noise_recs = build_pack("noise", pools["noise"], cfg.noise_hours, cfg) if pools["noise"] else []

spoken = sum(sum(b - a for a, b in r["seg"]) for r in speech_recs)
total_fr = sum(r["nf"] for r in speech_recs)
print(f"speech activity in source material: {spoken / max(total_fr, 1):.1%}")

In [ ]:
rirs = []
for p in pools["rir"]:
    h = load_audio(p, cfg.sr)
    if h is None or len(h) < 64:
        continue
    k = int(np.argmax(np.abs(h)))
    h = h[max(0, k - 16):k + int(0.30 * cfg.sr)]
    m = np.abs(h).max()
    if m > 1e-5:
        rirs.append((h / m).astype(np.float32))
print(f"{len(rirs)} impulse responses ready")

## Cross-checking the labels

Silero VAD runs over a quarter of the packed speech. Files where it disagrees
badly with the energy labeller get dropped, because that usually means the
"clean" source was not clean — an audiobook chapter with music under it, say.
Where the two roughly agree, speech Silero found _adjacent to_ an existing
segment gets merged in, which mostly recovers soft onsets.

Silero never becomes the sole source of truth. If it did, the model could not do
better than its teacher; used as a filter and a partial union, it removes bad
files without importing Silero's own errors wholesale.

Needs internet on. Without it this cell prints a warning and the energy labels
stand alone, which is a real but survivable quality loss.


In [ ]:
def open_mm(kind, wid, cfg, _cache={}):
    key = (kind, wid)
    if key not in _cache:
        _cache[key] = np.memmap(os.path.join(cfg.cache_dir, f"{kind}_{wid}.raw"),
                                dtype=np.int16, mode="r")
    return _cache[key]


def read_rec(kind, rec, cfg, start=0, n=None):
    mm = open_mm(kind, rec["w"], cfg)
    a = rec["o"] + start
    b = rec["o"] + rec["n"] if n is None else a + n
    return mm[a:min(b, rec["o"] + rec["n"])].astype(np.float32) / 32768.0


def load_silero():
    try:
        model, _ = torch.hub.load("snakers4/silero-vad", "silero_vad",
                                  trust_repo=True, onnx=False)
        return model.to(DEVICE).eval()
    except Exception as ex:
        print(f"silero unavailable ({ex}) - energy labels only")
        return None


@torch.no_grad()
def silero_mask(model, x, n_frames, cfg):
    t = torch.from_numpy(x).float().to(DEVICE)
    try:
        p = model.audio_forward(t.unsqueeze(0), sr=cfg.sr)[0].float().cpu().numpy()
    except Exception:
        model.reset_states()
        chunks = t[:len(t) // 512 * 512].view(-1, 512)
        p = torch.cat([model(c.unsqueeze(0), cfg.sr).view(-1) for c in chunks]).cpu().numpy()
    if len(p) < 2:
        return None
    src = np.arange(len(p)) * (512.0 / cfg.hop)
    return np.interp(np.arange(n_frames), src, p) > 0.5


teacher = load_silero() if cfg.use_teacher else None

if teacher is not None:
    rng = random.Random(cfg.seed)
    picked = [i for i in range(len(speech_recs)) if rng.random() < cfg.teacher_frac]
    ious, dropped, t0 = [], set(), time.time()

    for n, i in enumerate(picked):
        rec = speech_recs[i]
        lab = np.unpackbits(np.frombuffer(rec["lab"], np.uint8))[:rec["nf"]].astype(bool)
        try:
            tm = silero_mask(teacher, read_rec("speech", rec, cfg), rec["nf"], cfg)
        except Exception:
            continue
        if tm is None:
            continue
        union = (lab | tm).sum()
        iou = (lab & tm).sum() / max(union, 1)
        ious.append(iou)
        if iou < cfg.min_label_iou:
            dropped.add(i)
            continue
        merged = lab | (tm & dilate(lab, 20))
        merged = drop_short(close_gaps(merged, 10), 6)
        s, e = runs(merged)
        rec["seg"] = list(zip(s.tolist(), e.tolist()))
        rec["lab"] = np.packbits(merged).tobytes()
        if n % 2000 == 0 and n:
            print(f"  {n}/{len(picked)}  mean IoU {np.mean(ious):.3f}")

    speech_recs = [r for i, r in enumerate(speech_recs) if i not in dropped]
    del teacher
    torch.cuda.empty_cache()
    print(f"checked {len(ious)} files in {time.time() - t0:.0f}s | "
          f"mean IoU {np.mean(ious):.3f} | dropped {len(dropped)}")

speech_recs = [r for r in speech_recs if r.get("seg")]
print(f"{len(speech_recs)} speech files usable")

## Splits

Validation holds out whole _files_ on both sides, so val speakers and val noise
types are never seen in training. Splitting after augmentation would leak
badly — one source file feeds thousands of synthetic chunks, and a random split
over chunks would put the same speaker on both sides of it.


In [ ]:
def split(recs, frac, salt):
    rng = random.Random(cfg.seed + salt)
    idx = list(range(len(recs)))
    rng.shuffle(idx)
    cut = max(1, int(len(idx) * frac))
    hold = set(idx[:cut])
    return ([r for i, r in enumerate(recs) if i not in hold],
            [r for i, r in enumerate(recs) if i in hold])


tr_speech, va_speech = split(speech_recs, 0.04, 1)
tr_noise, va_noise = split(noise_recs, 0.06, 2) if noise_recs else ([], [])
tr_rir, va_rir = (rirs[:-100], rirs[-100:]) if len(rirs) > 400 else (rirs, rirs)

print(f"train {len(tr_speech)} speech / {len(tr_noise)} noise")
print(f"val   {len(va_speech)} speech / {len(va_noise)} noise")

## Building scenes

Each training example is a 4 s scene assembled on the fly: speech spans dropped
onto a canvas with realistic pauses, optionally run through a room response,
then mixed with one or two noise sources at a random SNR — measured over the
speech-active frames only, so the requested SNR means what it says regardless of
how much of the scene is silence.

Two details do most of the work:

- Spans start slightly _before_ a detected segment, so the model sees genuine
  onsets rather than hard cuts into full-volume speech.
- One chunk in eight contains no speech at all, and one in thirty is digital
  silence. That is what keeps the false alarm rate down on long pauses, which is
  the failure mode that actually annoys people using a VAD.

Validation uses `fixed`, which derives each scene's randomness from its index
instead of the worker seed. The val set is then identical across epochs and
across runs, and its SNR bucket is round-robin, which is what makes the per-SNR
table further down comparable between training runs.


In [ ]:
class ChunkSet(torch.utils.data.Dataset):
    def __init__(self, speech, noise, rirs, cfg, length, fixed=None):
        self.speech, self.noise, self.rirs = speech, noise, rirs
        self.cfg, self.length, self.fixed = cfg, length, fixed
        self.hop, self.sr = cfg.hop, cfg.sr
        self.T = cfg.chunk_samples
        self.F = cfg.chunk_frames
        self.rng = None
        self.labcache = {}

    def __len__(self):
        return self.length

    def _rand(self, i):
        if self.fixed is not None:
            return np.random.default_rng((self.fixed * 1000003 + i) & 0xFFFFFFFF)
        if self.rng is None:
            self.rng = np.random.default_rng(torch.initial_seed() % (2 ** 32))
        return self.rng

    def _labels(self, rec):
        key = (rec["w"], rec["o"])
        v = self.labcache.get(key)
        if v is None:
            v = np.unpackbits(np.frombuffer(rec["lab"], np.uint8))[:rec["nf"]].astype(bool)
            if len(self.labcache) > 4096:
                self.labcache.clear()
            self.labcache[key] = v
        return v

    def _span(self, rng, room):
        rec = self.speech[rng.integers(len(self.speech))]
        a, b = rec["seg"][rng.integers(len(rec["seg"]))]
        pre = int(rng.integers(0, 20))
        start = max(0, a - pre)
        want = int(rng.uniform(*self.cfg.span_range) * self.sr) // self.hop
        nfr = int(min(want, room, rec["nf"] - start))
        if nfr < 12:
            return None
        x = read_rec("speech", rec, self.cfg, start * self.hop, nfr * self.hop)
        lab = self._labels(rec)[start:start + nfr]
        if len(x) < nfr * self.hop:
            x = np.pad(x, (0, nfr * self.hop - len(x)))
        if len(lab) < nfr:
            lab = np.pad(lab, (0, nfr - len(lab)))
        return x, lab

    def _speech_scene(self, rng, dense):
        canvas = np.zeros(self.T, np.float32)
        lab = np.zeros(self.F, np.float32)
        cur = int(rng.integers(0, 60 if dense else 140))
        gaps = (0.02, 0.15) if dense else self.cfg.gap_range

        while cur < self.F - 12:
            got = self._span(rng, self.F - cur)
            if got is None:
                break
            x, l = got
            nfr = len(l)
            canvas[cur * self.hop:cur * self.hop + nfr * self.hop] += \
                x * 10 ** (rng.uniform(-7, 0) / 20)
            lab[cur:cur + nfr] = l
            cur += nfr + int(rng.uniform(*gaps) * self.sr) // self.hop
        return canvas, lab

    def _noise_bed(self, rng):
        out = np.zeros(self.T, np.float32)
        if not self.noise or rng.random() < self.cfg.p_synthetic_noise:
            return out + self._synthetic(rng)
        for _ in range(int(rng.integers(1, 3))):
            rec = self.noise[rng.integers(len(self.noise))]
            if rec["n"] <= self.T:
                y = read_rec("noise", rec, self.cfg)
                y = np.tile(y, self.T // max(len(y), 1) + 1)[:self.T]
            else:
                s = int(rng.integers(0, rec["n"] - self.T))
                y = read_rec("noise", rec, self.cfg, s, self.T)
            if len(y) < self.T:
                y = np.pad(y, (0, self.T - len(y)))
            out += y * 10 ** (rng.uniform(-9, 0) / 20)
        return out

    def _synthetic(self, rng):
        w = rng.standard_normal(self.T).astype(np.float32)
        kind = rng.integers(0, 3)
        if kind == 1:
            w = np.cumsum(w) / 40.0
        elif kind == 2:
            f = rng.uniform(50, 4000)
            w = 0.4 * w + np.sin(2 * np.pi * f * np.arange(self.T) / self.sr).astype(np.float32)
        return (w / (np.abs(w).max() + 1e-6) * 0.3).astype(np.float32)

    def __getitem__(self, i):
        rng = self._rand(i)
        cfg = self.cfg
        roll = rng.random()

        if roll < cfg.p_digital_silence:
            x = rng.standard_normal(self.T).astype(np.float32) * 1e-5
            bucket = i % len(VAL_SNRS) if self.fixed is not None else 0
            return torch.from_numpy(x), torch.zeros(self.F), bucket

        if roll < cfg.p_digital_silence + cfg.p_noise_only:
            speech, lab = np.zeros(self.T, np.float32), np.zeros(self.F, np.float32)
        else:
            dense = roll < cfg.p_digital_silence + cfg.p_noise_only + cfg.p_dense
            speech, lab = self._speech_scene(rng, dense)

        if self.rirs and lab.any() and rng.random() < cfg.p_reverb:
            h = self.rirs[rng.integers(len(self.rirs))]
            speech = fftconvolve(speech, h)[:self.T].astype(np.float32)

        noise = self._noise_bed(rng)

        if self.fixed is not None:
            snr_id = i % len(VAL_SNRS)
            snr = VAL_SNRS[snr_id]
        else:
            snr_id = 0
            snr = rng.uniform(*cfg.snr_range)

        if lab.any():
            active = np.repeat(lab > 0.5, self.hop)[:self.T]
            sp = float((speech[:len(active)][active] ** 2).mean() + 1e-10)
            npw = float((noise ** 2).mean() + 1e-10)
            gain = 0.0 if snr > 90 else math.sqrt(sp / (npw * 10 ** (snr / 10.0)))
            mix = speech + noise * gain
        else:
            mix = noise

        peak = np.abs(mix).max()
        if peak > 1e-6:
            mix = mix / peak * 10 ** (rng.uniform(*cfg.level_range) / 20.0)

        if rng.random() < cfg.p_lowpass:
            k = int(rng.integers(2, 4))
            mix = resample_poly(resample_poly(mix, 1, k), k, 1)[:self.T].astype(np.float32)
            if len(mix) < self.T:
                mix = np.pad(mix, (0, self.T - len(mix)))
        if rng.random() < cfg.p_clip:
            lim = np.abs(mix).max() * rng.uniform(0.3, 0.7)
            mix = np.clip(mix, -lim, lim)

        return torch.from_numpy(np.ascontiguousarray(mix, np.float32)), torch.from_numpy(lab), snr_id


train_set = ChunkSet(tr_speech, tr_noise, tr_rir, cfg, cfg.batch * cfg.steps_per_epoch)
val_set = ChunkSet(va_speech, va_noise, va_rir, cfg, cfg.val_chunks, fixed=7)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=cfg.batch, num_workers=cfg.workers, shuffle=False,
    pin_memory=True, drop_last=True, persistent_workers=True, prefetch_factor=4)
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=cfg.batch, num_workers=cfg.workers, shuffle=False, pin_memory=True)

xb, yb, _ = next(iter(val_loader))
print(xb.shape, yb.shape, f"speech frames {yb.mean():.1%}")

## Features

64 log-mel bins over the 30 ms / 10 ms grid, `center=False` so frame _t_ only
ever sees samples up to _t_. That last flag is what makes the model causal at
the front end; without it every frame would peek 15 ms into the future and the
streaming detector could not reproduce the offline result.

Normalisation uses fixed global statistics rather than per-utterance CMVN. Mean
and variance normalising an utterance requires the whole utterance, which a live
microphone does not have, so a model trained that way cannot be streamed
faithfully. The statistics are estimated once here and then baked into the graph
as buffers, so the backend does not have to know about them at all.


In [ ]:
mel_fn = torchaudio.transforms.MelSpectrogram(
    sample_rate=cfg.sr, n_fft=cfg.n_fft, win_length=cfg.win, hop_length=cfg.hop,
    f_min=cfg.fmin, f_max=cfg.fmax, n_mels=cfg.n_mels, center=False, power=2.0).to(DEVICE)

LOG_EPS = 1e-6


def to_mel(wav):
    return torch.log(mel_fn(wav) + LOG_EPS)


stat_loader = torch.utils.data.DataLoader(
    ChunkSet(tr_speech, tr_noise, tr_rir, cfg, cfg.batch * 40, fixed=99),
    batch_size=cfg.batch, num_workers=cfg.workers)

s1 = torch.zeros(cfg.n_mels, device=DEVICE)
s2 = torch.zeros(cfg.n_mels, device=DEVICE)
cnt = 0
with torch.no_grad():
    for w, _, _ in stat_loader:
        m = to_mel(w.to(DEVICE))
        s1 += m.sum((0, 2))
        s2 += (m ** 2).sum((0, 2))
        cnt += m.shape[0] * m.shape[2]

MEL_MEAN = (s1 / cnt).cpu()
MEL_STD = ((s2 / cnt - (s1 / cnt) ** 2).clamp_min(1e-6).sqrt()).cpu()
print(f"mel mean {MEL_MEAN.mean():.2f}  std {MEL_STD.mean():.2f}")
del stat_loader
gc.collect()

## Model

A causal MarbleNet: five residual stacks of 1D time-channel separable
convolutions with kernels 11 to 25, then a unidirectional GRU for unbounded past
context. Convolutions pad on the left only, so nothing reads ahead. Left
receptive field is 172 frames — about 1.7 s of history — and total algorithmic
latency is one window, 30 ms.

Separable convolutions are what keep this near 1 M parameters instead of 10 M at
the same accuracy: a depthwise convolution over time followed by a pointwise
mix across channels, rather than one dense kernel doing both. That matters when
the thing has to run on a CPU next to a web server, which is exactly where it
ends up here.

The GRU is the part the rule-based detectors have no equivalent of. Hysteresis
and hangover are a fixed, hand-set memory of the recent past; the GRU is a
learned one, and it is why the model can hold a decision through a 200 ms pause
in a sentence without holding it through a 200 ms pause between sentences.

> MarbleNet: Jia, Wang & Ginsburg, _MarbleNet: Deep 1D Time-Channel Separable
> Convolutional Neural Network for Voice Activity Detection_, ICASSP 2021.


In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, cin, cout, k, dilation=1, groups=1):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout, k, dilation=dilation, groups=groups, bias=False)
        self.left = (k - 1) * dilation

    def forward(self, x):
        return self.conv(F.pad(x, (self.left, 0)))


class SepBlock(nn.Module):
    def __init__(self, cin, cout, k, drop):
        super().__init__()
        self.dw = CausalConv1d(cin, cin, k, groups=cin)
        self.pw = nn.Conv1d(cin, cout, 1, bias=False)
        self.bn = nn.BatchNorm1d(cout)
        self.drop = nn.Dropout(drop)

    def forward(self, x, act=True):
        x = self.bn(self.pw(self.dw(x)))
        return self.drop(F.relu(x)) if act else x


class ResStack(nn.Module):
    def __init__(self, cin, cout, k, n, drop):
        super().__init__()
        self.blocks = nn.ModuleList(
            [SepBlock(cin if i == 0 else cout, cout, k, drop) for i in range(n)])
        self.skip = nn.Sequential(nn.Conv1d(cin, cout, 1, bias=False), nn.BatchNorm1d(cout))
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        y = x
        for i, b in enumerate(self.blocks):
            y = b(y, act=i < len(self.blocks) - 1)
        return self.drop(F.relu(y + self.skip(x)))


class VADNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        c = cfg.channels
        self.register_buffer("mel_mean", torch.zeros(cfg.n_mels))
        self.register_buffer("mel_std", torch.ones(cfg.n_mels))
        self.stem = nn.Sequential(CausalConv1d(cfg.n_mels, c, 5), nn.BatchNorm1d(c), nn.ReLU())
        self.stacks = nn.ModuleList(
            [ResStack(c, c, k, cfg.sub_blocks, cfg.dropout) for k in cfg.kernels])
        self.neck = nn.Sequential(CausalConv1d(c, c, 3, dilation=2), nn.BatchNorm1d(c), nn.ReLU())
        self.rnn = nn.GRU(c, cfg.gru_hidden, batch_first=True)
        self.head = nn.Linear(cfg.gru_hidden, 1)
        self.context = 4 + sum((k - 1) * cfg.sub_blocks for k in cfg.kernels) + 4

    def trunk(self, mel):
        x = (mel - self.mel_mean[None, :, None]) / self.mel_std[None, :, None]
        x = self.stem(x)
        for s in self.stacks:
            x = s(x)
        return self.neck(x)

    def forward(self, mel, h: Optional[torch.Tensor] = None):
        x, h = self.rnn(self.trunk(mel).transpose(1, 2), h)
        return self.head(x).squeeze(-1), h


model = VADNet(cfg).to(DEVICE)
model.mel_mean.copy_(MEL_MEAN.to(DEVICE))
model.mel_std.copy_(MEL_STD.to(DEVICE))
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f} M params | "
      f"left context {model.context} frames ({model.context * cfg.hop * 1000 // cfg.sr} ms)")

## Training

Frames within 2 of a label transition are down-weighted to 0.3. The exact onset
frame is genuinely ambiguous - the labeller dilated boundaries by 30 ms, and the
true answer is somewhere in there - so forcing the model to commit at the
boundary just adds noise to the gradient.

Mild frequency masking is the only spectral augmentation. Time masking is the
usual partner to it and is deliberately absent: blanking a span of time would
contradict the per-frame labels sitting directly on top of it.

An EMA of the weights is what gets evaluated and exported. It costs one extra
copy of the model in memory and reliably buys a little accuracy at no inference
cost.


In [ ]:
def frame_weights(y, cfg):
    d = (y[:, 1:] != y[:, :-1]).float()
    edge = F.pad(d, (1, 0))
    edge = F.max_pool1d(edge.unsqueeze(1), 2 * cfg.boundary_frames + 1,
                        stride=1, padding=cfg.boundary_frames).squeeze(1)
    return 1.0 - (1.0 - cfg.boundary_weight) * edge


def spec_augment(m, rng):
    fill = m.mean()
    n = m.shape[1]
    for _ in range(2):
        w = int(rng.integers(0, 13))
        if w:
            f0 = int(rng.integers(0, n - w))
            m[:, f0:f0 + w, :] = fill
    return m + torch.randn_like(m) * 0.05


class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float()
                       for k, v in model.state_dict().items() if v.dtype.is_floating_point}

    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                if k in self.shadow:
                    self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)

    def copy_to(self, model):
        sd = model.state_dict()
        model.load_state_dict({k: self.shadow[k].to(v.dtype) if k in self.shadow else v
                               for k, v in sd.items()})


@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    probs, gold, snrs = [], [], []
    for w, y, s in loader:
        mel = to_mel(w.to(DEVICE, non_blocking=True))
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logit, _ = net(mel)
        probs.append(torch.sigmoid(logit.float()).cpu())
        gold.append(y)
        snrs.append(s.view(-1, 1).expand(-1, y.shape[1]))
    return (torch.cat(probs).numpy().ravel(),
            torch.cat(gold).numpy().ravel(),
            torch.cat(snrs).numpy().ravel())


def quick_stats(p, y, thr=0.5):
    pred = p > thr
    g = y > 0.5
    tp = float((pred & g).sum())
    fp = float((pred & ~g).sum())
    fn = float((~pred & g).sum())
    tn = float((~pred & ~g).sum())
    return {
        "acc": (tp + tn) / len(y),
        "f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "fa": fp / max(fp + tn, 1),
        "miss": fn / max(fn + tp, 1),
    }

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=cfg.lr, total_steps=cfg.epochs * cfg.steps_per_epoch,
    pct_start=cfg.pct_warmup, div_factor=20, final_div_factor=200)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
ema = EMA(model, cfg.ema_decay)
aug_rng = np.random.default_rng(cfg.seed)

CKPT = os.path.join(cfg.out_dir, "vad_ckpt.pt")
start_epoch, best_f1 = 0, 0.0

if os.path.exists(CKPT):
    st = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(st["model"])
    opt.load_state_dict(st["opt"])
    sched.load_state_dict(st["sched"])
    scaler.load_state_dict(st["scaler"])
    ema.shadow = {k: v.to(DEVICE) for k, v in st["ema"].items()}
    start_epoch, best_f1 = st["epoch"] + 1, st["best_f1"]
    print(f"resumed at epoch {start_epoch}, best F1 {best_f1:.4f}")

budget_s = cfg.time_budget_h * 3600
wall0 = time.time()
stopped_early = False

for epoch in range(start_epoch, cfg.epochs):
    model.train()
    t0, run, seen = time.time(), 0.0, 0

    for step, (w, y, _) in enumerate(train_loader):
        w = w.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        wt = frame_weights(y, cfg)
        tgt = y * (1 - cfg.label_smooth) + 0.5 * cfg.label_smooth

        mel = spec_augment(to_mel(w), aug_rng)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logit, _ = model(mel)
            loss = (F.binary_cross_entropy_with_logits(logit.float(), tgt, reduction="none")
                    * wt).sum() / wt.sum()

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(opt)
        scaler.update()
        if sched.last_epoch < sched.total_steps - 1:
            sched.step()
        ema.update(model)

        run += loss.item()
        seen += 1
        if step % 300 == 0:
            print(f"  e{epoch} {step:5d}/{cfg.steps_per_epoch}  loss {run / max(seen, 1):.4f}  "
                  f"lr {sched.get_last_lr()[0]:.2e}  {(time.time() - wall0) / 60:.0f} min")
            run, seen = 0.0, 0

        if time.time() - wall0 > budget_s:
            stopped_early = True
            break

    backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
    ema.copy_to(model)
    p, g, _ = evaluate(model, val_loader)
    m = quick_stats(p, g)
    model.load_state_dict(backup)

    print(f"epoch {epoch}  {time.time() - t0:.0f}s  acc {m['acc']:.4f}  f1 {m['f1']:.4f}  "
          f"fa {m['fa']:.4f}  miss {m['miss']:.4f}")

    if m["f1"] > best_f1:
        best_f1 = m["f1"]
        torch.save({k: v.to(torch.float32) for k, v in ema.shadow.items()},
                   os.path.join(cfg.out_dir, "vad_best.pt"))
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                "ema": ema.shadow, "epoch": epoch, "best_f1": best_f1,
                "cfg": asdict(cfg)}, CKPT)

    if stopped_early:
        print(f"\ntime budget of {cfg.time_budget_h} h reached - stopping here.")
        print("The learning rate had not finished annealing, so this model is a "
              "little short of what the schedule would have given. Rerun the "
              "notebook to resume from the checkpoint and finish the schedule.")
        break

print(f"done in {(time.time() - wall0) / 3600:.2f} h, best val F1 {best_f1:.4f}")

## Results

Frame-level metrics on held-out speakers and held-out noise, broken down by
mixing SNR. The threshold sweep picks an operating point.

For most applications a slightly miss-averse point beats raw F1 — a clipped word
costs a user more than a little extra audio does — so the exported threshold is
nudged below the F1 optimum unless that costs more than a couple of points of
false alarm.


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

best = VADNet(cfg).to(DEVICE)
best.load_state_dict(torch.load(os.path.join(cfg.out_dir, "vad_best.pt"), map_location=DEVICE),
                     strict=False)
best.eval()

probs, gold, snr_ids = evaluate(best, val_loader)

auc = roc_auc_score(gold > 0.5, probs)
fpr, tpr, thr = roc_curve(gold > 0.5, probs)
eer_i = int(np.nanargmin(np.abs(fpr - (1 - tpr))))
print(f"ROC-AUC {auc:.5f}   EER {fpr[eer_i]:.4f} @ thr {thr[eer_i]:.3f}")

grid = np.arange(0.05, 0.96, 0.01)
scores = [quick_stats(probs, gold, t) for t in grid]
f1_t = float(grid[int(np.argmax([s["f1"] for s in scores]))])

best_t = f1_t
base_fa = quick_stats(probs, gold, f1_t)["fa"]
for t in np.arange(f1_t, 0.04, -0.01):
    s = quick_stats(probs, gold, t)
    if s["fa"] > base_fa + 0.02:
        break
    best_t = float(t)

m = quick_stats(probs, gold, best_t)
print(f"\nF1-optimal threshold {f1_t:.2f}, exported threshold {best_t:.2f}")
print(f"threshold {best_t:.2f}  acc {m['acc']:.4f}  f1 {m['f1']:.4f}  "
      f"fa {m['fa']:.4f}  miss {m['miss']:.4f}  DER {m['fa'] + m['miss']:.4f}")

print(f"\n{'condition':>10} {'acc':>8} {'f1':>8} {'FA':>8} {'miss':>8}")
for i, name in enumerate(SNR_NAMES):
    sel = snr_ids == i
    if sel.sum() == 0:
        continue
    s = quick_stats(probs[sel], gold[sel], best_t)
    print(f"{name:>10} {s['acc']:>8.4f} {s['f1']:>8.4f} {s['fa']:>8.4f} {s['miss']:>8.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(fpr, tpr, lw=2)
ax[0].plot([0, 1], [0, 1], "--", c="0.7", lw=1)
ax[0].set(xlabel="false alarm", ylabel="hit rate", title=f"ROC (AUC {auc:.4f})",
          xlim=(0, .2), ylim=(.8, 1))
ax[1].plot(grid, [s["f1"] for s in scores], label="F1")
ax[1].plot(grid, [s["fa"] for s in scores], label="false alarm")
ax[1].plot(grid, [s["miss"] for s in scores], label="miss")
ax[1].axvline(best_t, c="k", ls="--", lw=1)
ax[1].set(xlabel="threshold", title="operating point")
ax[1].legend()
plt.tight_layout()
plt.show()

## Post-processing

Raw per-frame decisions flicker, and the fix is the same one the rule-based
detectors use: hysteresis, then minimum durations. `backend/app/vad/pipeline.py`
already implements all of it — `apply_hysteresis_threshold`, `apply_hangover`,
`finalise_segments` — so the deployed detector reuses that code rather than
carrying its own copy. What is written below is the same thing in the notebook's
own idiom, so the numbers here and the numbers the backend reports agree.

Note how much smaller the correction is than it was for the energy detector.
The GRU has already done most of this work internally, which is why the exported
`min_speech_ms` and `min_silence_ms` are far shorter than the rule-based
defaults of 120 and 100 ms.


In [ ]:
HOP_S = cfg.hop / cfg.sr

ON_OFF_GAP = 0.10
MIN_SPEECH_S = 0.08
MIN_SILENCE_S = 0.10
PAD_S = 0.03


def smooth(prob, on=None, off=None, min_speech=MIN_SPEECH_S, min_sil=MIN_SILENCE_S,
           pad=PAD_S, hop_s=HOP_S):
    on = best_t + ON_OFF_GAP if on is None else on
    off = max(0.02, best_t - ON_OFF_GAP) if off is None else off
    m = hysteresis(prob, on, off)
    m = close_gaps(m, int(min_sil / hop_s))
    m = drop_short(m, int(min_speech / hop_s))
    return dilate(m, int(pad / hop_s))


def segments(prob, **kw):
    s, e = runs(smooth(prob, **kw))
    return [(round(a * HOP_S, 2), round(b * HOP_S, 2)) for a, b in zip(s, e)]


raw = probs.reshape(-1, cfg.chunk_frames)
ref = gold.reshape(-1, cfg.chunk_frames)
post = np.stack([smooth(r) for r in raw[:1024]])
ps = quick_stats(post.ravel().astype(float), ref[:1024].ravel(), 0.5)
pre = quick_stats(raw[:1024].ravel(), ref[:1024].ravel(), best_t)
print(f"before smoothing: acc {pre['acc']:.4f}  f1 {pre['f1']:.4f}  "
      f"fa {pre['fa']:.4f}  miss {pre['miss']:.4f}")
print(f"after  smoothing: acc {ps['acc']:.4f}  f1 {ps['f1']:.4f}  "
      f"fa {ps['fa']:.4f}  miss {ps['miss']:.4f}")

demo = int(np.argmax(ref[:1024].mean(1) * (ref[:1024].mean(1) < 0.8)))
t = np.arange(cfg.chunk_frames) * HOP_S
plt.figure(figsize=(12, 3))
plt.plot(t, raw[demo], lw=1, label="p(speech)")
plt.fill_between(t, 0, ref[demo], alpha=.2, step="mid", label="truth")
plt.step(t, post[demo], c="r", lw=1.2, where="mid", label="smoothed")
plt.axhline(best_t, c="k", ls=":", lw=1)
plt.legend(loc="upper right")
plt.xlabel("seconds")
plt.tight_layout()
plt.show()
print("segments:", segments(raw[demo]))

## Streaming

Two different kinds of memory have to be carried between blocks, and mixing them
up is the easy mistake here.

The convolution stack needs the previous `context` mel frames re-fed on every
call, because its output for a new frame genuinely depends on them. The GRU must
_not_ see those frames again — it already consumed them, and re-feeding them
would advance its state twice over the same audio. So the exported graph takes
`context + n_new` frames, runs the trunk over all of them, and hands only the
last `n_new` to the GRU.

That leaves what to seed the history with on the very first block. Offline, the
causal convolutions left-pad with zeros — but they pad _after_ normalisation, so
the equivalent seed is not a zero mel frame, it is `mel_mean`, which normalises
to exactly zero. Seeded that way the streamed output is numerically identical to
a full-file pass from the first frame onwards, with no warm-up period. The check
below asserts it rather than assuming it: a silent divergence would mean the
microphone tab and the file explorer disagree about the same audio.


In [ ]:
class StreamingVAD:
    def __init__(self, net, cfg, device="cpu"):
        self.net = net.eval().to(device)
        self.cfg, self.device = cfg, device
        self.context = net.context
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=cfg.sr, n_fft=cfg.n_fft, win_length=cfg.win, hop_length=cfg.hop,
            f_min=cfg.fmin, f_max=cfg.fmax, n_mels=cfg.n_mels, center=False,
            power=2.0).to(device)
        self.seed = net.mel_mean.detach().to(device).view(1, -1, 1).repeat(1, 1, self.context)
        self.reset()

    def reset(self):
        self.tail = torch.zeros(0, device=self.device)
        self.hist = self.seed.clone()
        self.h = None

    @torch.no_grad()
    def __call__(self, block):
        if not torch.is_tensor(block):
            block = torch.from_numpy(np.asarray(block, np.float32))
        self.tail = torch.cat([self.tail, block.float().to(self.device)])
        n = len(self.tail)
        if n < self.cfg.win:
            return np.zeros(0, np.float32)

        nf = 1 + (n - self.cfg.win) // self.cfg.hop
        need = (nf - 1) * self.cfg.hop + self.cfg.win
        m = torch.log(self.mel(self.tail[:need].unsqueeze(0)) + LOG_EPS)
        self.tail = self.tail[nf * self.cfg.hop:]

        full = torch.cat([self.hist, m], -1)
        self.hist = full[..., -self.context:]
        feat = self.net.trunk(full)[..., self.context:].transpose(1, 2)
        out, self.h = self.net.rnn(feat, self.h)
        return torch.sigmoid(self.net.head(out).squeeze(-1))[0].cpu().numpy()


probe = VADNet(cfg)
probe.load_state_dict(torch.load(os.path.join(cfg.out_dir, "vad_best.pt"), map_location="cpu"),
                      strict=False)
probe.eval()

stream = StreamingVAD(probe, cfg)
sig = val_set[3][0]

with torch.no_grad():
    offline_mel = torch.log(stream.mel(sig.unsqueeze(0)) + LOG_EPS)
    ref_out = torch.sigmoid(probe(offline_mel)[0]).numpy().ravel()

rng = np.random.default_rng(0)
pos, chunks = 0, []
while pos < len(sig):
    step = int(rng.integers(160, 4000))
    chunks.append(stream(sig[pos:pos + step]))
    pos += step
stream_out = np.concatenate(chunks)

n = min(len(ref_out), len(stream_out))
drift = np.abs(ref_out[:n] - stream_out[:n]).max()
print(f"streaming vs offline: max abs diff {drift:.2e}")
assert drift < 1e-4, "streaming path diverged from the offline path"

t0 = time.time()
stream.reset()
for i in range(0, len(sig), 1600):
    stream(sig[i:i + 1600])
rt = (len(sig) / cfg.sr) / (time.time() - t0)
print(f"single CPU thread: {rt:.0f}x realtime at 100 ms blocks")

## Export

Four files come out, all into `/kaggle/working` where the Output panel can
download them:

| File               | What                                                    |
| ------------------ | ------------------------------------------------------- |
| `vad.onnx`         | the graph — mel in, probability and GRU state out       |
| `vad_frontend.npz` | the analysis window, the mel filterbank, and `mel_mean` |
| `vad_meta.json`    | frame grid, thresholds, and the metrics above           |
| `vad_best.pt`      | the PyTorch weights, for further training               |

`vad_frontend.npz` is the part worth explaining. The backend has to compute the
same log-mel this notebook did, and "same" has to mean bit-for-bit, not
approximately. Rebuilding a mel filterbank from parameters is exactly where that
goes wrong quietly: torchaudio defaults to HTK-scaled, unnormalised filters,
librosa defaults to Slaney-scaled, area-normalised ones. Both are called "the
mel filterbank" and they are not the same matrix. Shipping the array removes the
question.


In [ ]:
CONTEXT = probe.context


class ExportWrapper(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net
        self.context = int(net.context)

    def forward(self, mel, h):
        feat = self.net.trunk(mel)[..., self.context:].transpose(1, 2)
        x, h = self.net.rnn(feat, h)
        return torch.sigmoid(self.net.head(x).squeeze(-1)), h


wrap = ExportWrapper(probe).eval()
dummy_mel = torch.randn(1, cfg.n_mels, CONTEXT + 100)
dummy_h = torch.zeros(1, 1, cfg.gru_hidden)

ONNX_PATH = os.path.join(cfg.out_dir, "vad.onnx")
ONNX_ARGS = dict(
    input_names=["mel", "h_in"], output_names=["prob", "h_out"],
    dynamic_axes={"mel": {0: "batch", 2: "time"}, "prob": {0: "batch", 1: "time"},
                  "h_in": {1: "batch"}, "h_out": {1: "batch"}},
    opset_version=17)

try:
    torch.onnx.export(wrap, (dummy_mel, dummy_h), ONNX_PATH, dynamo=False, **ONNX_ARGS)
except TypeError:                    # torch < 2.6 has no dynamo kwarg
    torch.onnx.export(wrap, (dummy_mel, dummy_h), ONNX_PATH, **ONNX_ARGS)

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip -q install onnxruntime")
    import onnxruntime as ort

_probe_sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
_ins = {i.name: i.shape for i in _probe_sess.get_inputs()}
_outs = {o.name: o.shape for o in _probe_sess.get_outputs()}
assert list(_ins) == ["mel", "h_in"], f"inputs came out as {list(_ins)}"
assert list(_outs) == ["prob", "h_out"], f"outputs came out as {list(_outs)}"
assert isinstance(_ins["mel"][2], str), f"mel time axis is frozen at {_ins['mel'][2]}"
assert isinstance(_outs["prob"][1], str), f"prob time axis is frozen at {_outs['prob'][1]}"
print(f"onnx  mel{_ins['mel']}  h_in{_ins['h_in']}  ->  "
      f"prob{_outs['prob']}  h_out{_outs['h_out']}")

try:
    torch.jit.save(torch.jit.script(wrap), os.path.join(cfg.out_dir, "vad.ts"))
except Exception as ex:
    torch.jit.save(torch.jit.trace(wrap, (dummy_mel, dummy_h)),
                   os.path.join(cfg.out_dir, "vad.ts"))
    print(f"scripted export fell back to tracing: {ex}")

def _buffers_of(mel):
    try:
        return mel.spectrogram.window, mel.mel_scale.fb
    except AttributeError:
        b = dict(mel.named_buffers())
        return (next(v for k, v in b.items() if k.endswith("window")),
                next(v for k, v in b.items() if k.endswith("fb")))


_w, _f = _buffers_of(mel_fn)
window = _w.detach().cpu().numpy().astype(np.float32)
fbank = _f.detach().cpu().numpy().astype(np.float32)

assert window.shape == (cfg.win,), f"window {window.shape}, expected ({cfg.win},)"
assert fbank.shape == (cfg.n_freqs, cfg.n_mels), \
    f"fbank {fbank.shape}, expected ({cfg.n_freqs}, {cfg.n_mels})"

mel_mean_np = probe.mel_mean.detach().cpu().numpy().astype(np.float32)

np.savez(os.path.join(cfg.out_dir, "vad_frontend.npz"),
         window=window, fbank=fbank, mel_mean=mel_mean_np)
print(f"window {window.shape}  fbank {fbank.shape}  mel_mean {mel_mean_np.shape}")

meta = {
    "sr": cfg.sr,
    "win": cfg.win, "hop": cfg.hop, "n_fft": cfg.n_fft,
    "frame_ms": cfg.win * 1000.0 / cfg.sr,
    "hop_ms": cfg.hop * 1000.0 / cfg.sr,
    "n_mels": cfg.n_mels, "fmin": cfg.fmin, "fmax": cfg.fmax,
    "center": False, "log_eps": LOG_EPS,
    "gru_hidden": cfg.gru_hidden,
    "left_context_frames": probe.context,
    "threshold": round(best_t, 3),
    "enter": round(min(0.95, best_t + ON_OFF_GAP), 3),
    "exit": round(max(0.02, best_t - ON_OFF_GAP), 3),
    "min_speech_ms": MIN_SPEECH_S * 1000.0,
    "min_silence_ms": MIN_SILENCE_S * 1000.0,
    "pad_ms": PAD_S * 1000.0,
    "params_m": round(sum(p.numel() for p in probe.parameters()) / 1e6, 3),
    "val_auc": round(float(auc), 5),
    "val_f1": round(float(m["f1"]), 5),
    "val_fa": round(float(m["fa"]), 5),
    "val_miss": round(float(m["miss"]), 5),
    "val_eer": round(float(fpr[eer_i]), 5),
    "val_by_snr": {
        name: {k: round(float(v), 5)
               for k, v in quick_stats(probs[snr_ids == i], gold[snr_ids == i], best_t).items()}
        for i, name in enumerate(SNR_NAMES) if (snr_ids == i).sum()
    },
    "trained_on": {"speech_hours": cfg.speech_hours, "noise_hours": cfg.noise_hours,
                   "rirs": len(rirs), "epochs_planned": cfg.epochs,
                   "stopped_early": bool(stopped_early)},
}
with open(os.path.join(cfg.out_dir, "vad_meta.json"), "w") as fh:
    json.dump(meta, fh, indent=2)

for f in ["vad_best.pt", "vad.ts", "vad.onnx", "vad_frontend.npz", "vad_meta.json"]:
    p = os.path.join(cfg.out_dir, f)
    if os.path.exists(p):
        print(f"{f:20s} {os.path.getsize(p) / 1e6:.2f} MB")

## Checking what the backend will actually run

Everything above ran through PyTorch and torchaudio. The backend runs neither-
it computes the log-mel in numpy and the network in ONNX Runtime. Two
substitutions, each of which could be subtly wrong in a way no crash would
reveal: a wrong filterbank convention shifts every feature slightly, and the
model would keep producing plausible-looking probabilities that are simply worse.

So both halves are checked against the originals here, and the numpy front end
below is character-for-character the one in `backend/app/vad/neural.py`. It
takes the frame matrix `frame_signal()` already builds, which is why the frame
grid was pinned to 30 ms / 10 ms at the top: no re-buffering, no second grid,
no drift between what the explorer plots and what the model saw.


In [ ]:
try:
    import onnxruntime as ort
except ImportError:
    os.system("pip -q install onnxruntime")
    import onnxruntime as ort


def log_mel_from_frames(frames, window, fbank, log_eps=1e-6):
    spec = np.abs(np.fft.rfft(frames * window, axis=-1)) ** 2
    return np.log(spec.astype(np.float32) @ fbank + log_eps).T


audio = val_set[5][0].numpy()
frames = frames_of(np.ascontiguousarray(audio), cfg.win, cfg.hop)
np_mel = log_mel_from_frames(frames, window, fbank, LOG_EPS)

with torch.no_grad():
    tt_mel = torch.log(mel_fn(torch.from_numpy(audio).to(DEVICE)) + LOG_EPS).cpu().numpy()

k = min(np_mel.shape[1], tt_mel.shape[1])
front_err = np.abs(np_mel[:, :k] - tt_mel[:, :k]).max()
print(f"numpy log-mel vs torchaudio : max abs diff {front_err:.3e}")
assert front_err < 1e-3, "the numpy front end does not reproduce the training front end"

sess = ort.InferenceSession(ONNX_PATH, providers=["CPUExecutionProvider"])
seed = np.repeat(mel_mean_np[:, None], CONTEXT, axis=1)


def zero_state():
    return np.zeros((1, 1, cfg.gru_hidden), np.float32)


primed = np.concatenate([seed, np_mel], axis=1)[None].astype(np.float32)
onnx_p, _ = sess.run(None, {"mel": primed, "h_in": zero_state()})

with torch.no_grad():
    torch_p = torch.sigmoid(probe(torch.from_numpy(np_mel[None]))[0]).numpy()

assert onnx_p.shape == torch_p.shape, f"{onnx_p.shape} vs {torch_p.shape}"
graph_err = np.abs(onnx_p - torch_p).max()
print(f"onnxruntime vs pytorch      : max abs diff {graph_err:.3e}")
assert graph_err < 1e-3, "the ONNX graph does not reproduce the PyTorch model"

h = zero_state()
hist = seed.copy()
pending = np.zeros(0, np.float32)
out = []
rng = np.random.default_rng(1)

pos = 0
while pos < len(audio):
    block = audio[pos:pos + int(rng.integers(160, 4000))]
    pos += len(block)
    buf = np.concatenate([pending, block])
    if len(buf) < cfg.win:
        pending = buf
        continue
    nf = 1 + (len(buf) - cfg.win) // cfg.hop
    fr = frames_of(np.ascontiguousarray(buf), cfg.win, cfg.hop)
    pending = buf[nf * cfg.hop:]

    mel = log_mel_from_frames(fr, window, fbank, LOG_EPS)
    full = np.concatenate([hist, mel], axis=1)
    hist = full[:, -CONTEXT:]
    p, h = sess.run(None, {"mel": full[None].astype(np.float32), "h_in": h})
    out.append(p[0])

streamed = np.concatenate(out)
k = min(len(streamed), torch_p.shape[1])
e2e = np.abs(streamed[:k] - torch_p[0][:k]).max()
print(f"onnx streamed vs offline    : max abs diff {e2e:.3e}  over {k} frames")
assert e2e < 1e-3, "the streamed ONNX path diverged"
print("\nall three parity checks passed - the export is faithful")